In [ ]:
import numpy as np
from itertools import compress

In [ ]:
class Cell():
    def __init__(self, x_coord: int, y_coord: int, color_id: int):
        self.x = x_coord
        self.y = y_coord
        self.color_id = color_id
        self.filled = False
        self.queen = False

class Chunk():
    def __init__(self, color_id: int):
        self.color_id = color_id
        self.cells = []
        self.coords = []
    
    def add_cell(self, cell: Cell):
        self.cells.append(cell)
        self.coords.append((cell.x, cell.y))

    @property
    def has_queen(self):
        return any(cell.queen for cell in self.cells)
    
    @property
    def free_cells(self):
        """Returns a list of cells that are neither filled nor have a queen."""
        return [cell for cell in self.cells if not cell.filled and not cell.queen]
    @property
    def queen_count(self):
        return sum(1 for cell in self.cells if cell.queen)
    
    def get_coords(self):
        return [(cell.x, cell.y) for cell in self.cells]


In [ ]:
# make the grid

grid_size = 8 # always a square for queens
num_colors = 8 # is equal to grid size for all puzzles i think

chunks = {color_id: Chunk(color_id) for color_id in range(num_colors)}

grid_cells = []

for i in range(grid_size):
    row = []

    for j in range(grid_size):
        color_id = get_color_from_map(i,j) #TODO: make that cv part
        new_cell = Cell(i, j, color_id)
        chunks[color_id].add_cell(new_cell)
        row.append(new_cell)

    grid_cells.append(row)

In [ ]:
# method for filling around where queens are placed

def mark_neighbors_filled(grid_cells, row, col, grid_size):
    # Iterate through the 3x3 area centered on the queen
    for i in range(max(0, row - 1), min(grid_size, row + 2)):
        for j in range(max(0, col - 1), min(grid_size, col + 2)):
            grid_cells[i][j].filled = True

In [ ]:
# if a chunk only has 1 free cell put a queen in it

for color_id, chunk in chunks.items():
    if chunk.has_queen():
        continue
    
    available = chunk.free_cells 

    if len(available) == 1:
        q_cell = available[0]
        q_cell.queen = True
        
        # 1. Mark all cells in the same ROW as filled
        for j in range(grid_size):
            if j != q_cell.y:
                grid_cells[q_cell.x][j].filled = True
                
        # 2. Mark all cells in the same COLUMN as filled
        for i in range(grid_size):
            if i != q_cell.x:
                grid_cells[i][q_cell.y].filled = True
                
        mark_neighbors_filled(grid_cells, q_cell.x, q_cell.y, grid_size)

In [ ]:
# check if any chunks only have free cells in a row or column
# and fill all other cells in that row or column if so

for color_id, chunk in chunks.items():
    if chunk.has_queen():
        continue

    available = chunk.free_cells
    if not available: 
        continue

    # 1. Check if all available cells share the same Row 
    first_x = available[0].x
    if all(c.x == first_x for c in available):
        # Every possible spot for this color is in row 'first_x'
        # So, no OTHER color can have a queen in row 'first_x'
        for j in range(grid_size):
            target_cell = grid_cells[first_x][j]
            if target_cell.color_id != color_id:
                target_cell.filled = True

    # 2. Check if all available cells share the same Column
    first_y = available[0].y
    if all(c.y == first_y for c in available):
        # Every possible spot for this color is in column 'first_y'
        for i in range(grid_size):
            target_cell = grid_cells[i][first_y]
            if target_cell.color_id != color_id:
                target_cell.filled = True

In [ ]:
# if a row or column only has free cells of 1 color, fill the rest of that color chunk

# 1. Iterate through every Row
for r in range(grid_size):
    # Get all free cells in this specific row
    free_in_row = [grid_cells[r][c] for c in range(grid_size) 
                   if not grid_cells[r][c].filled and not grid_cells[r][c].queen]
    
    if not free_in_row:
        continue

    # Check if all free cells in this row belong to the same Chunk (color_id)
    first_color = free_in_row[0].color_id
    if all(cell.color_id == first_color for cell in free_in_row):
        # CLAIM: The queen for this row MUST be in this chunk.
        # Therefore, any cell in this chunk that ISN'T in this row can be filled.
        target_chunk = chunks[first_color]
        for cell in target_chunk.cells:
            if cell.x != r: # If the cell is in the chunk but NOT in this row
                cell.filled = True

# 2. Iterate through every Column
for c in range(grid_size):
    # Get all free cells in this specific column
    free_in_col = [grid_cells[r][c] for r in range(grid_size) 
                   if not grid_cells[r][c].filled and not grid_cells[r][c].queen]
    
    if not free_in_col:
        continue

    # Check if all free cells in this column belong to the same Chunk
    first_color = free_in_col[0].color_id
    if all(cell.color_id == first_color for cell in free_in_col):
        # CLAIM: The queen for this column MUST be in this chunk.
        # Fill any cell in this chunk that ISN'T in this column.
        target_chunk = chunks[first_color]
        for cell in target_chunk.cells:
            if cell.y != c: # If the cell is in the chunk but NOT in this column
                cell.filled = True

In [ ]:
# check if a cell is excluded by a chunk where all free cells are adjascent
for x in range(grid_size):
    for y in range(grid_size):
        target_cell = grid_cells[x][y]
        
        if target_cell.filled or target_cell.queen:
            continue

        for color_id, chunk in chunks.items():
            if chunk.has_queen():
                continue

            available = chunk.free_cells
            if not available:
                continue
            
            def is_adjacent(c1, c2):
                return abs(c1.x - c2.x) <= 1 and abs(c1.y - c2.y) <= 1

            if all(is_adjacent(cell, target_cell) for cell in available):
                if target_cell.color_id != color_id:
                    target_cell.filled = True